# 03. CF-KAN и контроль MLP-AE
Класс `KANAutoencoder` берётся из `jindeok/CF-KAN` без изменений. Его ускоренный потомок `FastKANAE` даёт тот же результат
(проверяется ниже). Вход: мульти-хот вектор истории, цель: восстановить его. Для test вход = train + val.

Ограничения, которые нужно указать в тексте работы:
- скрининг гиперпараметров неравный: у CF-KAN три итерации с разными сетками, у MLP-AE одна сетка из 8 вариантов, каждый вариант 1500 шагов (около 1.4 эпохи);
- при бинарном входе первый слой KAN эквивалентен `Linear(N → H)`, поэтому сравнение CF-KAN и MLP-AE проверяет вклад сплайнового декодера, а не «KAN вообще».

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gc
import importlib
import json
import os
import sys
import time

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    default_work_dir = '/content/drive/MyDrive/code2repo/kan_rec'
except ImportError:
    default_work_dir = os.path.abspath('./kan_rec')

WORK_DIR = os.environ.get('KAN_REC_DIR', default_work_dir)
THIRD_PARTY_DIR = os.path.join(os.path.dirname(WORK_DIR), 'third_party')
CFKAN_DIR = os.path.join(THIRD_PARTY_DIR, 'CF-KAN')
DATASET = 'ml1m'  # 'amazon' | 'ml1m'
RUN_SCREENING = False  # результаты скрининга для Amazon уже лежат в results/

sys.path.insert(0, WORK_DIR)
import common
importlib.reload(common)

common.clone_repo('https://github.com/jindeok/CF-KAN', CFKAN_DIR)
sys.path.insert(0, CFKAN_DIR)
from models.DNN import KANAutoencoder

paths = common.dataset_paths(WORK_DIR, DATASET)
results_dir = paths['results']
os.makedirs(results_dir, exist_ok=True)
device = common.check_gpu()

CF-KAN: 57d38ce
Tesla T4, 14.6 GiB, sm_75, torch 2.11.0+cu130


## Данные

In [3]:
data = common.load_data(paths['data'])
n_users, n_items = data['n_users'], data['n_items']


def build_interaction_matrix(include_val):
    rows, cols = [], []
    for user, items in enumerate(data['train']):
        items = list(items)
        if include_val:
            items.append(int(data['val'][user]))
        rows.extend([user] * len(items))
        cols.extend(item - 1 for item in items)  # id айтема 1..N → колонка 0..N-1
    values = np.ones(len(rows), dtype=np.float32)
    return sp.csr_matrix((values, (rows, cols)), shape=(n_users, n_items))


train_matrix = build_interaction_matrix(include_val=False)
test_matrix = build_interaction_matrix(include_val=True)
print(f'users={n_users:,} items={n_items:,} nnz train={train_matrix.nnz:,}')


def dense_batch(matrix, users):
    return torch.from_numpy(matrix[users].toarray()).float().to(device)

users=6,040 items=3,416 nnz train=987,531


## Модели
`FastKANAE` хранит те же параметры, что `KANAutoencoder`, но первый слой считается быстрее. Вход бинарный, поэтому
`phi_i(x) = phi_i(0) + x * (phi_i(1) - phi_i(0))` выполняется точно: сплайны считаются один раз за шаг, а вклад
пользователей получается умножением `X @ D`.

`center=True` — отклонение от оригинала. Отсутствующие айтемы вносят 0 вместо `phi_i(0)`, константа заменяется обучаемым смещением.
Без этого сумма `phi_i(0)` по всем входам одинакова для всех пользователей и заглушает индивидуальный сигнал.

In [4]:
class MLPAE(nn.Module):
    """Контроль без KAN: тот же автокодировщик на обычных линейных слоях."""

    def __init__(self, n_items, hidden, dropout):
        super().__init__()
        self.encoder = nn.Linear(n_items, hidden)
        self.decoder = nn.Linear(hidden, n_items)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.decoder(F.silu(self.dropout(self.encoder(x))))

    def regularization_loss(self):
        return 0.0


class FastKANAE(KANAutoencoder):
    def __init__(self, input_dim, hidden_dims, grid_size=2, spline_order=3, dropout_rate=0.5,
                 out_dropout=True, center=False):
        super().__init__(input_dim, hidden_dims, grid_size, spline_order, dropout_rate)
        self.out_dropout = out_dropout  # True как в оригинале: dropout и на выходных логитах
        self.center = center
        if center:
            self.enc_bias = nn.Parameter(torch.zeros(hidden_dims[0]))

    def _phi(self):
        layer = self.encoder_layers[0]
        points = torch.tensor([[0.], [1.]], device=layer.base_weight.device)
        points = points.expand(2, layer.in_features).contiguous()
        spline_part = torch.einsum('rnc,onc->rno', layer.b_splines(points), layer.scaled_spline_weight)
        base_part = layer.base_activation(points).unsqueeze(-1) * layer.base_weight.t().unsqueeze(0)
        return spline_part + base_part

    def _first_layer(self, x):
        phi = self._phi()
        slope = phi[1] - phi[0]
        offset = self.enc_bias if self.center else phi[0].sum(0)
        return offset + x @ slope

    def forward(self, x):
        x = self.dropout(self._first_layer(x))
        for layer in self.encoder_layers[1:]:
            x = self.dropout(layer(x))
        for i, layer in enumerate(self.decoder_layers):
            x = layer(x)
            is_last = i == len(self.decoder_layers) - 1
            if self.out_dropout or not is_last:
                x = self.dropout(x)
        return x


def build_model(cfg):
    if cfg.get('arch', 'kan') == 'mlp':
        return MLPAE(n_items, cfg['hidden'], cfg['dropout']).to(device)
    model = FastKANAE(n_items, [cfg['hidden']], cfg['grid'], 3, cfg['dropout'],
                      cfg.get('out_dropout', True), cfg.get('center', False))
    return model.to(device)

## Проверка эквивалентности
Выходы и градиенты `FastKANAE` должны совпасть с `KANAutoencoder` с точностью около 1e-6.

In [5]:
torch.manual_seed(0)
reference = KANAutoencoder(n_items, [32], 2, 3, 0.5).to(device)
fast = FastKANAE(n_items, [32], 2, 3, 0.5).to(device)
fast.load_state_dict(reference.state_dict())
reference.eval()
fast.eval()

sample = dense_batch(train_matrix, np.arange(16))
out_reference, out_fast = reference(sample), fast(sample)
out_reference.pow(2).mean().backward()
out_fast.pow(2).mean().backward()

grad_diff = max((p.grad - q.grad).abs().max().item()
                for p, q in zip(reference.parameters(), fast.parameters()))
print(f'выход: max|diff|={(out_reference - out_fast).abs().max().item():.2e} '
      f'(масштаб {out_reference.abs().max().item():.2e}), градиенты: max|diff|={grad_diff:.2e}')

# center=True с bias = sum phi(0) должен воспроизводить оригинал
centered = FastKANAE(n_items, [32], 2, 3, 0.5, center=True).to(device)
centered.load_state_dict(reference.state_dict(), strict=False)
with torch.no_grad():
    centered.enc_bias.copy_(fast._phi()[0].sum(0))
centered.eval()
print(f'center=True и bias=sum phi(0): max|diff|={(centered(sample) - out_reference).abs().max().item():.2e}')

del reference, fast, centered, out_reference, out_fast
gc.collect()
torch.cuda.empty_cache()

/content/drive/MyDrive/code2repo/third_party/CF-KAN/models/DNN.py:68: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.base_weight)


выход: max|diff|=1.79e-07 (масштаб 4.32e-01), градиенты: max|diff|=3.49e-10
center=True и bias=sum phi(0): max|diff|=1.79e-07


## Обучение и оценка

In [6]:
def release_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def reconstruction_loss(output, target, kind):
    if kind == 'mse':
        return F.mse_loss(output, target)
    return -(target * F.log_softmax(output, dim=1)).sum(1).mean()


def training_loss(model, reconstruction, cfg):
    if cfg['reg'] > 0:
        return reconstruction + cfg['reg'] * model.regularization_loss()
    return reconstruction


def make_score_fn(model):
    def score_fn(batch, split):
        matrix = train_matrix if split == 'val' else test_matrix
        scores = model(dense_batch(matrix, batch))
        return torch.cat([scores.new_zeros(len(batch), 1), scores], dim=1)  # колонка 0 = padding
    return score_fn


def train_run(cfg, seed, max_epochs, patience, eval_every, eval_users, log_path=None):
    torch.manual_seed(seed)
    rng = np.random.RandomState(seed)
    model = build_model(cfg)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    score_fn = make_score_fn(model)

    best_ndcg, best_epoch, bad_evals, best_state = -1, 0, 0, None
    log, epoch_times, stopped = [], [], 'max_epochs'

    for epoch in range(1, max_epochs + 1):
        model.train()
        started = time.time()
        order = rng.permutation(n_users)
        epoch_loss, n_batches = 0.0, 0

        for start in range(0, n_users, cfg['batch']):
            batch = dense_batch(train_matrix, order[start:start + cfg['batch']])
            reconstruction = reconstruction_loss(model(batch), batch, cfg['loss'])
            loss = training_loss(model, reconstruction, cfg)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_loss += reconstruction.item()
            n_batches += 1

        epoch_times.append(time.time() - started)
        message = f'epoch {epoch:3d}  rec_loss {epoch_loss / n_batches:.5f}  ({epoch_times[-1]:.0f}s)'

        if epoch % eval_every == 0 or epoch == max_epochs:
            model.eval()
            val = common.evaluate(score_fn, data, 'val', batch_size=256, device=device, users=eval_users)
            message += f"  | val R@10 {val['Recall@10']:.4f}  N@10 {val['NDCG@10']:.4f}  R@20 {val['Recall@20']:.4f}"
            log.append(dict(epoch=epoch, rec_loss=epoch_loss / n_batches, **val))
            if log_path:
                with open(log_path, 'w') as f:
                    json.dump(log, f, indent=1)

            if val['NDCG@10'] > best_ndcg:
                best_ndcg, best_epoch, bad_evals = val['NDCG@10'], epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad_evals += 1

        print(message)
        if bad_evals >= patience:
            print('early stop')
            stopped = 'early_stop'
            break

    del model, optimizer, score_fn
    release_gpu_memory()
    return dict(best_state=best_state, best_epoch=best_epoch, log=log, n_params=n_params,
                sec_per_epoch=float(np.mean(epoch_times)), epochs_run=epoch, stopped=stopped)

## Скрининг (`RUN_SCREENING`)
Фиксированное число шагов, на отметках печатаются loss реконструкции и метрики val.
Для CF-KAN loss сравнивается с ориентирами: пока он не опустился заметно ниже уровня «по популярности», модель не учится.

In [7]:
def screen_run(cfg, steps, eval_at, eval_users, seed=0):
    torch.manual_seed(seed)
    rng = np.random.RandomState(seed)
    model = build_model(cfg)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    score_fn = make_score_fn(model)

    order, position = rng.permutation(n_users), 0
    recent_losses, curve, started = [], [], time.time()

    for step in range(1, steps + 1):
        if position + cfg['batch'] > n_users:
            order, position = rng.permutation(n_users), 0
        batch = dense_batch(train_matrix, order[position:position + cfg['batch']])
        position += cfg['batch']

        model.train()
        reconstruction = reconstruction_loss(model(batch), batch, cfg['loss'])
        loss = training_loss(model, reconstruction, cfg)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        recent_losses.append(reconstruction.item())

        if step in eval_at:
            model.eval()
            val = common.evaluate(score_fn, data, 'val', batch_size=256, device=device, users=eval_users)
            curve.append(dict(step=step, rec_loss=float(np.mean(recent_losses)), **val))
            recent_losses = []
            print(f"  шаг {step:5d} ({(time.time() - started) / 60:.1f} мин): rec_loss={curve[-1]['rec_loss']:.4f} "
                  f"R@10={val['Recall@10']:.4f} N@10={val['NDCG@10']:.4f} R@20={val['Recall@20']:.4f}")

    del model, optimizer
    release_gpu_memory()
    return curve


if RUN_SCREENING:
    HIDDEN, BATCH, GRID, DROPOUT = 256, 256, 2, 0.5
    STEPS, EVAL_AT = 1500, (500, 1000, 1500)
    screening_users = np.random.RandomState(1).choice(n_users, 5000, replace=False)

    item_counts = np.asarray(train_matrix.sum(0)).ravel()
    item_probs = item_counts / item_counts.sum()
    ce_uniform = train_matrix.sum(1).mean() * np.log(n_items)
    ce_popularity = -(train_matrix @ np.log(np.maximum(item_probs, 1e-12))).mean()
    print(f'ориентиры для rec_loss (CE): равномерно {ce_uniform:.1f}, по популярности {ce_popularity:.1f}')

    variants = [
        dict(center=True, loss='ce', lr=1e-4, reg=0.0, out_dropout=False),
        dict(center=True, loss='ce', lr=3e-4, reg=0.0, out_dropout=False),
        dict(center=True, loss='ce', lr=1e-3, reg=0.0, out_dropout=False),
        dict(center=True, loss='ce', lr=3e-4, reg=0.0, out_dropout=False, dropout=0.2),
    ]
    results = []
    for variant in variants:
        print('вариант:', variant)
        cfg = dict(hidden=HIDDEN, batch=BATCH, grid=GRID, dropout=DROPOUT)
        cfg.update(variant)
        results.append(dict(cfg=variant, curve=screen_run(cfg, STEPS, EVAL_AT, screening_users)))

    with open(f'{results_dir}/cfkan_screening_v3.json', 'w') as f:
        json.dump(results, f, indent=1)

## Конфигурации полных запусков
Равный бюджет у всех моделей: одинаковые `MAX_EPOCHS` и `PATIENCE`, валидация каждую эпоху на 30 тыс. пользователей, лимита по времени нет.
Эпоха CF-KAN на Amazon занимает около 5 минут, так что весь блок идёт много часов. Готовые запуски пропускаются, прерванный стартует заново.
`ALLOW_NO_RANKS=True` не перезапускает готовые запуски без файла рангов (они не попадут в бутстреп).

Гиперпараметры (lr, dropout) подбираются коротким скринингом: фиксированное число шагов, выбор по NDCG@10 на последней отметке.
Результаты лежат в `results*/mlpae_screening.json` и `results*/cfkan_screening.json`; если файл есть, скрининг не повторяется.
Для Amazon конфигурация CF-KAN зафиксирована (результат прежних итераций скрининга), для ML-1M обе модели подбираются
по одной сетке и с одинаковым бюджетом. Число шагов для ML-1M подобрано грубо (в эпохах это намного больше, чем 1500 шагов на Amazon):
посмотрите кривые в выводе и при необходимости поправьте `SCREEN_STEPS`.

In [8]:
SEEDS = [0, 1, 2, 3, 4]
ORIG_SEEDS = [0]  # оригинальный CF-KAN (MSE) считаем на одном seed
MAX_EPOCHS = dict(amazon=100, ml1m=300)[DATASET]
PATIENCE = 4
EVAL_EVERY = 1
VAL_USERS = 30000
ALLOW_NO_RANKS = True

SCREEN_STEPS = dict(amazon=1500, ml1m=600)[DATASET]
SCREEN_EVAL_AT = tuple(SCREEN_STEPS * i // 3 for i in (1, 2, 3))
SCREEN_USERS = 5000
LEARNING_RATES = (3e-4, 1e-3)
DROPOUTS = (0.2, 0.5)

CENTERED = dict(hidden=256, batch=256, grid=2, dropout=0.2, lr=3e-4, reg=0.0, loss='ce',
                out_dropout=False, center=True)
# оригинальная конфигурация: MSE + L1-регуляризация сплайнов, dropout и на логитах, без центрирования
ORIGINAL = dict(hidden=256, batch=256, grid=2, dropout=0.5, lr=1e-4, reg=0.3, loss='mse',
                out_dropout=True, center=False)


def load_or_run_screening(path, grid):
    if not os.path.exists(path):
        users = np.random.RandomState(1).choice(n_users, min(SCREEN_USERS, n_users), replace=False)
        results = []
        for cfg in grid:
            print('вариант:', {k: cfg[k] for k in ('hidden', 'lr', 'dropout')})
            curve = screen_run(cfg, SCREEN_STEPS, SCREEN_EVAL_AT, users)
            results.append(dict(cfg=cfg, curve=curve))
        with open(path, 'w') as f:
            json.dump(results, f, indent=1)
    with open(path) as f:
        return json.load(f)


def final_ndcg(entry):
    return entry['curve'][-1]['NDCG@10']


# MLP-AE: для каждого H (256 — то же скрытое пространство, 1792 — то же число параметров, что у KAN) лучший вариант
mlp_grid = [dict(arch='mlp', hidden=hidden, batch=256, loss='ce', reg=0.0, lr=lr, dropout=dropout)
            for hidden in (256, 1792) for lr in LEARNING_RATES for dropout in DROPOUTS]
mlp_results = load_or_run_screening(f'{results_dir}/mlpae_screening.json', mlp_grid)

best_mlp = {}
for entry in mlp_results:
    hidden = entry['cfg']['hidden']
    if hidden not in best_mlp or final_ndcg(entry) > final_ndcg(best_mlp[hidden]):
        best_mlp[hidden] = entry

for hidden, entry in best_mlp.items():
    chosen = {k: entry['cfg'][k] for k in ('lr', 'dropout')}
    print(f'MLP-AE H={hidden}: {chosen}, N@10 на скрининге {final_ndcg(entry):.4f}')

# CF-KAN (centered, CE)
if DATASET == 'amazon':
    centered_cfg = CENTERED
else:
    kan_grid = [dict(CENTERED, lr=lr, dropout=dropout) for lr in LEARNING_RATES for dropout in DROPOUTS]
    kan_results = load_or_run_screening(f'{results_dir}/cfkan_screening.json', kan_grid)
    centered_cfg = max(kan_results, key=final_ndcg)['cfg']
print('CF-KAN (centered, CE):', {k: centered_cfg[k] for k in ('lr', 'dropout')})

вариант: {'hidden': 256, 'lr': 0.0003, 'dropout': 0.2}
  шаг   200 (0.0 мин): rec_loss=1216.5865 R@10=0.0510 N@10=0.0266 R@20=0.0844
  шаг   400 (0.0 мин): rec_loss=1159.5159 R@10=0.0524 N@10=0.0269 R@20=0.0932
  шаг   600 (0.0 мин): rec_loss=1142.0623 R@10=0.0570 N@10=0.0280 R@20=0.0976
вариант: {'hidden': 256, 'lr': 0.0003, 'dropout': 0.5}
  шаг   200 (0.0 мин): rec_loss=1228.4765 R@10=0.0522 N@10=0.0261 R@20=0.0850
  шаг   400 (0.0 мин): rec_loss=1173.0095 R@10=0.0534 N@10=0.0271 R@20=0.0916
  шаг   600 (0.0 мин): rec_loss=1156.7682 R@10=0.0548 N@10=0.0272 R@20=0.0906
вариант: {'hidden': 256, 'lr': 0.001, 'dropout': 0.2}
  шаг   200 (0.0 мин): rec_loss=1173.1377 R@10=0.0586 N@10=0.0293 R@20=0.1018
  шаг   400 (0.0 мин): rec_loss=1121.7632 R@10=0.0684 N@10=0.0351 R@20=0.1238
  шаг   600 (0.0 мин): rec_loss=1103.7653 R@10=0.0734 N@10=0.0370 R@20=0.1316
вариант: {'hidden': 256, 'lr': 0.001, 'dropout': 0.5}
  шаг   200 (0.0 мин): rec_loss=1188.3850 R@10=0.0560 N@10=0.0276 R@20=0.0910
  

/content/drive/MyDrive/code2repo/third_party/CF-KAN/models/DNN.py:68: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.base_weight)


  шаг   200 (0.0 мин): rec_loss=1167.8374 R@10=0.0646 N@10=0.0318 R@20=0.1082
  шаг   400 (0.1 мин): rec_loss=1112.8901 R@10=0.0760 N@10=0.0379 R@20=0.1270
  шаг   600 (0.1 мин): rec_loss=1090.6137 R@10=0.0806 N@10=0.0393 R@20=0.1378
вариант: {'hidden': 256, 'lr': 0.0003, 'dropout': 0.5}
  шаг   200 (0.0 мин): rec_loss=1181.4956 R@10=0.0578 N@10=0.0285 R@20=0.0968
  шаг   400 (0.1 мин): rec_loss=1134.3737 R@10=0.0604 N@10=0.0308 R@20=0.1094
  шаг   600 (0.1 мин): rec_loss=1119.1262 R@10=0.0660 N@10=0.0333 R@20=0.1180
вариант: {'hidden': 256, 'lr': 0.001, 'dropout': 0.2}
  шаг   200 (0.0 мин): rec_loss=1132.5182 R@10=0.0782 N@10=0.0383 R@20=0.1356
  шаг   400 (0.1 мин): rec_loss=1072.4336 R@10=0.0848 N@10=0.0402 R@20=0.1434
  шаг   600 (0.1 мин): rec_loss=1047.4396 R@10=0.0814 N@10=0.0388 R@20=0.1452
вариант: {'hidden': 256, 'lr': 0.001, 'dropout': 0.5}
  шаг   200 (0.0 мин): rec_loss=1153.9813 R@10=0.0692 N@10=0.0338 R@20=0.1128
  шаг   400 (0.1 мин): rec_loss=1110.6954 R@10=0.0724 N@1

In [9]:
def can_skip(stem, cfg):
    path = f'{results_dir}/{stem}.json'
    if not os.path.exists(path):
        return False
    if common.load_run(results_dir, stem).get('cfg') != cfg:
        print(f'{stem}: конфигурация в файле отличается, переношу в _stale')
        common.move_to_stale(results_dir, stem)
        return False
    return common.is_finished(results_dir, stem, need_ranks=not ALLOW_NO_RANKS)


def run_full(cfg, model_name, stem, seed):
    val_users = np.random.RandomState(seed).choice(n_users, size=min(VAL_USERS, n_users), replace=False)
    run = train_run(cfg, seed, MAX_EPOCHS, PATIENCE, EVAL_EVERY, val_users,
                    log_path=f'{results_dir}/{stem}_log.json')

    model = build_model(cfg)
    model.load_state_dict(run['best_state'])
    model.eval()
    val_metrics, test_metrics, val_ranks, test_ranks = common.full_evaluation(make_score_fn(model), data, device)
    print('best epoch:', run['best_epoch'], '| stopped:', run['stopped'])
    print('VAL ', {k: round(v, 4) for k, v in val_metrics.items()})
    print('TEST', {k: round(v, 4) for k, v in test_metrics.items()})

    output = dict(
        model=model_name, seed=seed, cfg=cfg, max_epochs=MAX_EPOCHS, patience=PATIENCE, eval_every=EVAL_EVERY,
        best_epoch=run['best_epoch'], epochs_run=run['epochs_run'], stopped=run['stopped'],
        n_params=run['n_params'], sec_per_epoch=run['sec_per_epoch'], gpu=common.gpu_name(),
        val=val_metrics, test=test_metrics, log=run['log'],
    )
    common.save_run(results_dir, stem, output, val_ranks, test_ranks)
    del model, run
    release_gpu_memory()

## Запуск: CF-KAN (centered, CE), MLP-AE, затем оригинальный CF-KAN (MSE)

In [10]:
jobs = [(f'cfkan_ce_seed{seed}', 'CF-KAN (ce)', centered_cfg, seed) for seed in SEEDS]
for hidden, entry in best_mlp.items():
    jobs += [(f'mlpae_h{hidden}_seed{seed}', f'MLP-AE (H={hidden})', entry['cfg'], seed) for seed in SEEDS]
jobs += [(f'cfkan_orig_mse_seed{seed}', 'CF-KAN (mse)', ORIGINAL, seed) for seed in ORIG_SEEDS]

for stem, model_name, cfg, seed in jobs:
    if can_skip(stem, cfg):
        print(f'[пропуск] {stem}')
        continue
    print(f'\n=== {stem} ({model_name}) ===')
    run_full(cfg, model_name, stem, seed)
print('\nготово')


=== cfkan_ce_seed0 (CF-KAN (ce)) ===
epoch   1  rec_loss 1258.15677  (0s)  | val R@10 0.0404  N@10 0.0195  R@20 0.0695
epoch   2  rec_loss 1199.69160  (0s)  | val R@10 0.0497  N@10 0.0241  R@20 0.0810
epoch   3  rec_loss 1172.83816  (0s)  | val R@10 0.0546  N@10 0.0272  R@20 0.0921
epoch   4  rec_loss 1164.50246  (0s)  | val R@10 0.0558  N@10 0.0277  R@20 0.0957
epoch   5  rec_loss 1150.08916  (0s)  | val R@10 0.0576  N@10 0.0291  R@20 0.1000
epoch   6  rec_loss 1146.99535  (0s)  | val R@10 0.0588  N@10 0.0291  R@20 0.1061
epoch   7  rec_loss 1136.49971  (0s)  | val R@10 0.0609  N@10 0.0308  R@20 0.1060
epoch   8  rec_loss 1127.16692  (0s)  | val R@10 0.0639  N@10 0.0319  R@20 0.1109
epoch   9  rec_loss 1128.77171  (0s)  | val R@10 0.0656  N@10 0.0325  R@20 0.1151
epoch  10  rec_loss 1123.90425  (1s)  | val R@10 0.0680  N@10 0.0339  R@20 0.1177


KeyboardInterrupt: 